In [1]:
pip install pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

df = pd.read_csv("F:\Fourth_project\swiggy.csv")  # Replace with your actual file name
print(df.head())
print(df.info())


       id               name    city rating     rating_count   cost  \
0  567335     AB FOODS POINT  Abohar     --  Too Few Ratings  ₹ 200   
1  531342  Janta Sweet House  Abohar    4.4      50+ ratings  ₹ 200   
2  158203  theka coffee desi  Abohar    3.8     100+ ratings  ₹ 100   
3  187912          Singh Hut  Abohar    3.7      20+ ratings  ₹ 250   
4  543530      GRILL MASTERS  Abohar     --  Too Few Ratings  ₹ 250   

                      cuisine          lic_no  \
0            Beverages,Pizzas  22122652000138   
1               Sweets,Bakery  12117201000112   
2                   Beverages  22121652000190   
3            Fast Food,Indian  22119652000167   
4  Italian-American,Fast Food  12122201000053   

                                                link  \
0  https://www.swiggy.com/restaurants/ab-foods-po...   
1  https://www.swiggy.com/restaurants/janta-sweet...   
2  https://www.swiggy.com/restaurants/theka-coffe...   
3  https://www.swiggy.com/restaurants/singh-hut-n...  

In [3]:
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)  # or use fillna()

df.to_csv("cleaned_data.csv", index=False)


In [5]:
!pip install scikit-learn


  Using cached scikit_learn-1.6.1-cp310-cp310-win_amd64.whl (11.1 MB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl (41.3 MB)
  Using cached joblib-1.5.1-py3-none-any.whl (307 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)



[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Example: Keep top N frequent values only
top_names = df['name'].value_counts().nlargest(100).index
df['name'] = df['name'].where(df['name'].isin(top_names), 'other')

top_cities = df['city'].value_counts().nlargest(50).index
df['city'] = df['city'].where(df['city'].isin(top_cities), 'other')

top_cuisines = df['cuisine'].value_counts().nlargest(50).index
df['cuisine'] = df['cuisine'].where(df['cuisine'].isin(top_cuisines), 'other')


In [9]:
pip install category_encoders


     ---------------------------------------- 0.0/85.7 kB ? eta -:--:--
     ------------------ ------------------- 41.0/85.7 kB 653.6 kB/s eta 0:00:01
     -------------------------------------- 85.7/85.7 kB 802.9 kB/s eta 0:00:00
     ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
     ---------------------------------------- 0.1/9.8 MB 871.5 kB/s eta 0:00:12
     ---------------------------------------- 0.1/9.8 MB 653.6 kB/s eta 0:00:15
     ---------------------------------------- 0.1/9.8 MB 737.3 kB/s eta 0:00:14
      --------------------------------------- 0.1/9.8 MB 653.6 kB/s eta 0:00:15
      --------------------------------------- 0.2/9.8 MB 610.0 kB/s eta 0:00:16
      --------------------------------------- 0.2/9.8 MB 544.7 kB/s eta 0:00:18
      --------------------------------------- 0.2/9.8 MB 621.6 kB/s eta 0:00:16
      --------------------------------------- 0.2/9.8 MB 621.6 kB/


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from category_encoders import HashingEncoder

encoder = HashingEncoder(cols=categorical_cols, n_components=50)
encoded_df = encoder.fit_transform(df[categorical_cols])


In [11]:
from sklearn.preprocessing import LabelEncoder

for col in categorical_cols:
    df[col] = LabelEncoder().fit_transform(df[col])


In [12]:
from sklearn.preprocessing import OneHotEncoder
import pickle

# Select categorical columns for encoding
categorical_cols = ['name', 'city', 'cuisine']
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_array = encoder.fit_transform(df[categorical_cols])

# Save encoder
with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

# Create dataframe
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(categorical_cols))
numeric_df = df[['rating', 'rating_count', 'cost']].reset_index(drop=True)

final_df = pd.concat([encoded_df, numeric_df], axis=1)
final_df.to_csv("encoded_data.csv", index=False)


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

# Load data
encoded_df = pd.read_csv("encoded_data.csv")
original_df = pd.read_csv("cleaned_data.csv")

def get_recommendations(user_input_vector, top_n=5):
    similarities = cosine_similarity([user_input_vector], encoded_df)
    top_indices = similarities[0].argsort()[-top_n:][::-1]
    return original_df.iloc[top_indices]
